# Drosophila Neural Simulation → Flybody Integration

This notebook:
1. Runs a spiking neural network simulation of the Drosophila connectome (138,639 LIF neurons, 15M synapses)
2. Simulates **2 seconds of spontaneous activity**, then **stimulates sensory neuron subsets**
3. Records spikes from **descending motor neurons** (P9, DNa01/02, MDN, Giant Fiber, MN9, aDN1)
4. Maps motor neuron activity to **flybody** actions using CPG-modulated behavioral primitives
5. Records a **video** of the fly body moving in response to the neural simulation

**Prerequisites:**
- SONATA circuit files in `./sonata_circuit/` (run `convert_to_sonata.py` first)
- `torch`, `h5py`, `dm_control`, `flybody` installed

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Ensure flybody is importable
FLYBODY_PATH = Path("../../../flybody")
if str(FLYBODY_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(FLYBODY_PATH.resolve()))

CIRCUIT_DIR = Path("./sonata_circuit")
OUTPUT_DIR = Path("../../../obi-output/drosophila_flybody")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Experiment Configuration

Choose which sensory neurons to stimulate after the 2-second spontaneous period.
Each scenario activates different descending neuron populations, producing different behaviors.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Timing
# ──────────────────────────────────────────────────────────────────────
SPONTANEOUS_MS = 2000.0   # 2 s spontaneous activity (no input)
STIMULATED_MS  = 2000.0   # 2 s with sensory stimulation
TOTAL_MS = SPONTANEOUS_MS + STIMULATED_MS

# ──────────────────────────────────────────────────────────────────────
# Descending motor neurons (from fly-brain/code/paper-phil-drosophila/example.ipynb)
# These are the output neurons whose spikes drive the flybody.
# ──────────────────────────────────────────────────────────────────────
MOTOR_NEURONS = {
    # Forward walking
    "P9_left":      720575940627652358,
    "P9_right":     720575940635872101,
    "P9_oDN1_left": 720575940626730883,
    "P9_oDN1_right":720575940620300308,
    # Turning
    "DNa01_left":   720575940644438551,
    "DNa01_right":  720575940627787609,
    "DNa02_left":   720575940604737708,
    "DNa02_right":  720575940629327659,
    # Backward walking / startle
    "MDN_1":        720575940616026939,
    "MDN_2":        720575940631082808,
    "MDN_3":        720575940640331472,
    "MDN_4":        720575940610236514,
    # Escape
    "Giant_Fiber_1":720575940622838154,
    "Giant_Fiber_2":720575940632499757,
    # Feeding (proboscis)
    "MN9_left":     720575940660219265,
    "MN9_right":    720575940618238523,
    # Grooming
    "aDN1_left":    720575940624319124,
    "aDN1_right":   720575940616185531,
}

# ──────────────────────────────────────────────────────────────────────
# Sensory neuron groups for stimulation
# ──────────────────────────────────────────────────────────────────────
SENSORY_GROUPS = {
    "P9s": {
        "ids": [720575940627652358, 720575940635872101],
        "rate_hz": 100.0,
        "description": "P9 forward-walking neurons (100 Hz) → forward locomotion",
    },
    "sugar_GRNs": {
        "ids": [
            720575940616885538, 720575940630233916, 720575940639332736,
            720575940632889389, 720575940617000768, 720575940632425919,
            720575940637568838, 720575940629176663, 720575940621502051,
            720575940638202345, 720575940612670570, 720575940611875570,
            720575940621754367, 720575940633143833, 720575940613601698,
            720575940630797113, 720575940639198653, 720575940639259967,
            720575940624963786, 720575940640649691, 720575940610788069,
            720575940623172843, 720575940628853239,
        ],
        "rate_hz": 200.0,
        "description": "Sugar gustatory receptor neurons (200 Hz) → feeding",
    },
    "LC4s": {
        "ids": [
            720575940605598892, 720575940611134833, 720575940612580977,
            720575940613256863, 720575940613260959, 720575940614914107,
            720575940615462587, 720575940617176321, 720575940617266722,
            720575940618807105, 720575940620795728, 720575940622108001,
            720575940624017251, 720575940625038090, 720575940625934973,
            720575940625991043, 720575940626605200, 720575940626626895,
            720575940628454522, 720575940628462340, 720575940630851036,
            720575940638496720, 720575940603637438, 720575940610522009,
            720575940612093351, 720575940612323025, 720575940612380723,
            720575940612498129, 720575940612518055, 720575940612968421,
            720575940613609484, 720575940613638041, 720575940614572742,
            720575940614582946, 720575940615053580, 720575940615127227,
            720575940615232217, 720575940615575007, 720575940616066705,
            720575940616713355, 720575940617026260, 720575940617348379,
            720575940618002644, 720575940618234704, 720575940618234715,
            720575940618266459, 720575940618267227, 720575940618275520,
            720575940618312606, 720575940618676440, 720575940618709158,
            720575940618723749, 720575940619397542, 720575940620314221,
            720575940620314612, 720575940620731380, 720575940620903551,
            720575940621145821, 720575940621522458, 720575940621753579,
            720575940622330582, 720575940622531767, 720575940622939836,
            720575940624111763, 720575940624790781, 720575940624856762,
            720575940625841351, 720575940625845447, 720575940625906702,
            720575940625932421, 720575940626553596, 720575940626916936,
            720575940627519107, 720575940628064260, 720575940628081541,
            720575940628419527, 720575940628518400, 720575940628599895,
            720575940628606713, 720575940628699560, 720575940628891863,
            720575940629753807, 720575940629964591, 720575940630154660,
            720575940630484495, 720575940630998339, 720575940631032657,
            720575940631338271, 720575940632475449, 720575940632715234,
            720575940632769180, 720575940633013355, 720575940633218863,
            720575940633580384, 720575940634517856, 720575940635835967,
            720575940636957006, 720575940638456227, 720575940639817947,
            720575940640612480, 720575940641213824, 720575940645821316,
            720575940649229433, 720575940652611745,
        ],
        "rate_hz": 200.0,
        "description": "LC4 looming-sensitive visual neurons (200 Hz) → escape",
    },
}

# ──────────────────────────────────────────────────────────────────────
# ▶ SELECT EXPERIMENT HERE
# Options: "P9s", "sugar_GRNs", "LC4s", or combine e.g. ["P9s", "LC4s"]
# ──────────────────────────────────────────────────────────────────────
ACTIVE_EXPERIMENTS = ["P9s", "LC4s"]

print("Experiment configuration:")
print(f"  Spontaneous period: 0 – {SPONTANEOUS_MS:.0f} ms")
print(f"  Stimulated period:  {SPONTANEOUS_MS:.0f} – {TOTAL_MS:.0f} ms")
for exp_name in ACTIVE_EXPERIMENTS:
    g = SENSORY_GROUPS[exp_name]
    print(f"  → {exp_name}: {len(g['ids'])} neurons at {g['rate_hz']} Hz")
    print(f"    {g['description']}")

## 2. Load SONATA Circuit

In [ ]:
def load_sonata_circuit(circuit_dir: Path) -> dict:
    """Load SONATA circuit data from HDF5 files."""
    net = circuit_dir / "network"
    data = {}

    with open(circuit_dir / "circuit_config.json") as f:
        data["circuit_config"] = json.load(f)
    with open(circuit_dir / "simulation_config.json") as f:
        data["sim_config"] = json.load(f)

    # Internal nodes
    with h5py.File(net / "internal_nodes.h5", "r") as f:
        pop = f["nodes/internal"]
        data["n_neurons"] = len(pop["node_id"])
        data["flywire_ids"] = pop["0/flywire_id"][:]

    # Internal edges (recurrent synapses)
    with h5py.File(net / "internal_internal_edges.h5", "r") as f:
        pop = f["edges/internal_to_internal"]
        data["pre_idx"] = pop["source_node_id"][:].astype(np.int64)
        data["post_idx"] = pop["target_node_id"][:].astype(np.int64)
        data["weights"] = pop["0/syn_weight"][:]
        data["delays"] = pop["0/delay"][:]

    # Neuron dynamics parameters
    params_path = circuit_dir / "components" / "cell_models" / "flywire_lif.json"
    with open(params_path) as f:
        data["neuron_params"] = json.load(f)

    data["dt"] = data["sim_config"]["run"]["dt"]

    # Build FlyWire ID ↔ node_id maps
    data["fw_to_node"] = {int(fw): i for i, fw in enumerate(data["flywire_ids"])}
    data["node_to_fw"] = {i: int(fw) for i, fw in enumerate(data["flywire_ids"])}

    return data


t0 = perf_counter()
circuit = load_sonata_circuit(CIRCUIT_DIR)
n = circuit["n_neurons"]
dt = circuit["dt"]
print(f"Loaded in {perf_counter() - t0:.1f}s")
print(f"  Neurons: {n:,}")
print(f"  Synapses: {len(circuit['weights']):,}")
print(f"  dt: {dt} ms")

## 3. Resolve Neuron IDs to Node Indices

Map FlyWire IDs from the experiment config to SONATA node indices.

In [ ]:
fw_to_node = circuit["fw_to_node"]

# Resolve motor neuron FlyWire IDs → node indices
motor_node_ids = {}
for name, fw_id in MOTOR_NEURONS.items():
    if fw_id in fw_to_node:
        motor_node_ids[name] = fw_to_node[fw_id]
    else:
        print(f"  WARNING: {name} (fw={fw_id}) not found in circuit")

print(f"Motor neurons resolved: {len(motor_node_ids)}/{len(MOTOR_NEURONS)}")

# Resolve stimulation targets
stim_node_ids = {}  # name → list of node indices
stim_rates = {}     # name → rate in Hz
for exp_name in ACTIVE_EXPERIMENTS:
    group = SENSORY_GROUPS[exp_name]
    nodes = []
    for fw_id in group["ids"]:
        if fw_id in fw_to_node:
            nodes.append(fw_to_node[fw_id])
    stim_node_ids[exp_name] = nodes
    stim_rates[exp_name] = group["rate_hz"]
    print(f"  {exp_name}: {len(nodes)}/{len(group['ids'])} neurons at {group['rate_hz']} Hz")

# Flatten all stimulation targets
all_stim_nodes = []
all_stim_rates_hz = []
for exp_name in ACTIVE_EXPERIMENTS:
    for nid in stim_node_ids[exp_name]:
        all_stim_nodes.append(nid)
        all_stim_rates_hz.append(stim_rates[exp_name])

## 4. Build PyTorch Spiking Network Model

LIF neurons with alpha-function synapses, matching the SONATA/FlyWire circuit dynamics.

In [ ]:
class PoissonSpikeGenerator(nn.Module):
    def __init__(self, dt, scale, device="cpu"):
        super().__init__()
        self.prob_scale = dt / 1000.0
        self.scale = scale
        self.device = device

    def forward(self, rates):
        return torch.bernoulli(rates * self.prob_scale) * self.scale


class AlphaSynapse(nn.Module):
    def __init__(self, size, dt, tau_syn, delay_ms, device="cpu"):
        super().__init__()
        self.time_factor = dt / tau_syn
        self.steps_delay = int(delay_ms / dt)
        self.size = size
        self.device = device

    def state_init(self):
        conductance = torch.zeros(self.size, device=self.device)
        delay_buffer = torch.zeros(
            self.steps_delay + 1, self.size, device=self.device
        )
        return conductance, delay_buffer

    def forward(self, input_, conductance, delay_buffer, refrac):
        conductance_new = (
            conductance * (1 - self.time_factor) + delay_buffer[0, :] * refrac
        )
        delay_buffer = torch.roll(delay_buffer, shifts=-1, dims=0)
        delay_buffer[-1, :] = input_
        return conductance_new, delay_buffer


class LIFNeuron(nn.Module):
    def __init__(self, size, dt, params, device="cpu"):
        super().__init__()
        self.size = size
        self.time_factor = dt / params["tauMem"]
        self.v_reset = params["vReset"]
        self.v_rest = params["vRest"]
        self.v_threshold = params["vThreshold"]
        self.v_0 = params["v0"]
        self.device = device

    def state_init(self):
        v = torch.zeros(self.size, device=self.device) + self.v_0
        spikes = torch.zeros(self.size, device=self.device)
        return spikes, v

    def forward(self, input_current, v):
        v = v + self.time_factor * (input_current - (v - self.v_rest))
        spike = (v > self.v_threshold).float()
        reset = ((v - self.v_reset) * spike).detach()
        v = v - reset
        return spike, v


class AlphaLIF(nn.Module):
    def __init__(self, size, dt, params, device="cpu"):
        super().__init__()
        self.synapse = AlphaSynapse(
            size, dt, params["tauSyn"], params["tDelay"], device=device
        )
        self.neuron = LIFNeuron(size, dt, params, device=device)
        self.steps_refrac = int(params["tRefrac"] / dt)

    def state_init(self):
        conductance, delay_buffer = self.synapse.state_init()
        spikes, v = self.neuron.state_init()
        refrac = self.steps_refrac + torch.zeros_like(v)
        return conductance, delay_buffer, spikes, v, refrac

    def forward(self, input_, conductance, delay_buffer, spikes, v, refrac):
        refrac = refrac * (1 - spikes) + 1
        conductance_new, delay_buffer = self.synapse(
            input_, conductance, delay_buffer,
            (refrac > self.steps_refrac).float(),
        )
        spikes, v_new = self.neuron(conductance, v)
        conductance_reset = (conductance_new * spikes).detach()
        conductance_new = conductance_new - conductance_reset
        return conductance_new, delay_buffer, spikes, v_new, refrac


class SonataModel(nn.Module):
    def __init__(self, size, dt, params, weights_sparse, device="cpu"):
        super().__init__()
        self.neurons = AlphaLIF(size, dt, params, device=device)
        self.weights = weights_sparse
        self.poisson = PoissonSpikeGenerator(
            dt, params["scalePoisson"], device=device
        )
        self.w_scale = params["wScale"]

    def state_init(self):
        return self.neurons.state_init()

    def forward(self, rates, conductance, delay_buffer, spikes, v, refrac):
        spikes_input = self.poisson(rates)
        weighted_spikes = torch.mv(self.weights, spikes)
        conductance, delay_buffer, spikes, v, refrac = self.neurons(
            self.w_scale * (spikes_input + weighted_spikes),
            conductance, delay_buffer, spikes, v, refrac,
        )
        return conductance, delay_buffer, spikes, v, refrac

In [ ]:
# Map SONATA neuron params → PyTorch model params
np_ = circuit["neuron_params"]
stim_weight = 68.75  # wScale * scalePoisson = 0.275 * 250

params = {
    "tauSyn": np_["tau_syn_ex"],
    "tDelay": float(circuit["delays"][0]),
    "v0": np_["E_L"],
    "vReset": np_["V_reset"],
    "vRest": np_["E_L"],
    "vThreshold": np_["V_th"],
    "tauMem": np_["tau_m"],
    "tRefrac": np_["t_ref"],
    "scalePoisson": stim_weight,
    "wScale": 1.0,
}

num_steps = int(TOTAL_MS / dt)
stim_onset_step = int(SPONTANEOUS_MS / dt)

# Build sparse weight matrix
print("Building sparse weight matrix...")
t0 = perf_counter()
weight_coo = torch.sparse_coo_tensor(
    torch.tensor(
        np.array([circuit["post_idx"], circuit["pre_idx"]]),
        dtype=torch.long,
    ),
    torch.tensor(circuit["weights"], dtype=torch.float32),
    size=(n, n),
).coalesce()
weights_sparse = weight_coo.to_sparse_csr().to(DEVICE)
print(f"  Built in {perf_counter() - t0:.2f}s  ({n:,}×{n:,}, {len(circuit['weights']):,} nnz)")

model = SonataModel(n, dt, params, weights_sparse, device=DEVICE)
print(f"\nSimulation: {TOTAL_MS:.0f} ms ({num_steps:,} steps)")
print(f"  Spontaneous:  0 – {SPONTANEOUS_MS:.0f} ms (step 0 – {stim_onset_step:,})")
print(f"  Stimulated:   {SPONTANEOUS_MS:.0f} – {TOTAL_MS:.0f} ms (step {stim_onset_step:,} – {num_steps:,})")

## 5. Run Two-Phase Neural Simulation

**Phase 1** (0 → 2 s): No external input — spontaneous network dynamics.  
**Phase 2** (2 → 4 s): Poisson stimulation of selected sensory neurons.  

We record spikes from **all neurons** (sparse), with special focus on the 18 motor neurons.

In [ ]:
# Prepare stimulation rates tensor (activated at stim_onset_step)
stim_rates_tensor = torch.zeros(n, device=DEVICE)
for nid, rate in zip(all_stim_nodes, all_stim_rates_hz):
    stim_rates_tensor[nid] = rate

rates_off = torch.zeros(n, device=DEVICE)  # Phase 1: no input

# Reverse lookup: node_id → motor neuron name (O(1) per spike)
node_to_motor = {nid: name for name, nid in motor_node_ids.items()}

# Storage: per-motor-neuron spike timestep lists
motor_spike_trains = {name: [] for name in motor_node_ids}
all_spike_times = []
all_spike_ids = []

# Initialize model state
conductance, delay_buffer, spikes, v, refrac = model.state_init()

print(f"Simulating {TOTAL_MS:.0f} ms ({num_steps:,} steps)...")
t_sim = perf_counter()

with torch.no_grad():
    for t_step in range(num_steps):
        # Switch input at stimulation onset
        rates = rates_off if t_step < stim_onset_step else stim_rates_tensor

        conductance, delay_buffer, spikes, v, refrac = model(
            rates, conductance, delay_buffer, spikes, v, refrac
        )

        spike_mask = spikes > 0
        if spike_mask.any():
            n_idx = spike_mask.nonzero(as_tuple=True)[0].cpu().numpy()
            t_ms = t_step * dt
            all_spike_times.extend([t_ms] * len(n_idx))
            all_spike_ids.extend(n_idx.tolist())

            # Record motor neuron spikes via reverse lookup
            for nid in n_idx:
                motor_name = node_to_motor.get(int(nid))
                if motor_name is not None:
                    motor_spike_trains[motor_name].append(t_step)

        if num_steps >= 10000 and (t_step + 1) % (num_steps // 10) == 0:
            elapsed = perf_counter() - t_sim
            pct = (t_step + 1) / num_steps * 100
            phase = "spontaneous" if t_step < stim_onset_step else "stimulated"
            print(f"  {pct:3.0f}% ({t_step+1:,}/{num_steps:,}) - {elapsed:.1f}s [{phase}]")

sim_time = perf_counter() - t_sim
all_spike_times = np.array(all_spike_times)
all_spike_ids = np.array(all_spike_ids, dtype=np.int64)

n_total = len(all_spike_times)
n_active = len(set(all_spike_ids))
print(f"\nDone in {sim_time:.1f}s  (realtime: {TOTAL_MS/1000/sim_time:.3f}x)")
print(f"  Total spikes: {n_total:,}  |  Active neurons: {n_active:,}/{n:,}")

# Report motor neuron activity
print("\nMotor neuron spikes:")
for name, steps in motor_spike_trains.items():
    n_spk = len(steps)
    rate = n_spk / (TOTAL_MS / 1000) if n_spk > 0 else 0
    print(f"  {name:20s}: {n_spk:4d} spikes ({rate:.1f} Hz)")

## 6. Compute Motor Neuron Activation Traces

Convert discrete spikes into smooth activation signals by convolving with an exponential kernel.  
Then group neurons by behavioral function and compute aggregate activations.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Exponential smoothing of spike trains → continuous activation
# ──────────────────────────────────────────────────────────────────────
TAU_SMOOTH_MS = 100.0  # Exponential decay time constant

def spike_steps_to_activation(spike_steps, num_steps, dt, tau_ms):
    """Convert spike timestep indices to a smooth activation trace."""
    decay = np.exp(-dt / tau_ms)
    trace = np.zeros(num_steps)
    activation = 0.0
    spike_set = set(spike_steps)
    for t in range(num_steps):
        activation *= decay
        if t in spike_set:
            activation += 1.0 / (tau_ms / 1000.0)  # Normalize so integral ≈ rate
        trace[t] = activation
    return trace


# Compute per-neuron activation traces
motor_activations = {}
for name, steps in motor_spike_trains.items():
    motor_activations[name] = spike_steps_to_activation(
        steps, num_steps, dt, TAU_SMOOTH_MS
    )

# ──────────────────────────────────────────────────────────────────────
# Group motor neurons by behavioral function → aggregate activations
# ──────────────────────────────────────────────────────────────────────
BEHAVIOR_GROUPS = {
    "forward":  ["P9_left", "P9_right", "P9_oDN1_left", "P9_oDN1_right"],
    "turn_left":  ["DNa01_left", "DNa02_left"],
    "turn_right": ["DNa01_right", "DNa02_right"],
    "backward": ["MDN_1", "MDN_2", "MDN_3", "MDN_4"],
    "escape":   ["Giant_Fiber_1", "Giant_Fiber_2"],
    "feeding":  ["MN9_left", "MN9_right"],
    "grooming": ["aDN1_left", "aDN1_right"],
}

behavior_activations = {}
for beh, neuron_names in BEHAVIOR_GROUPS.items():
    traces = [motor_activations[n] for n in neuron_names if n in motor_activations]
    if traces:
        behavior_activations[beh] = np.mean(traces, axis=0)
    else:
        behavior_activations[beh] = np.zeros(num_steps)

# Normalize each behavior activation to [0, 1] based on peak
for beh in behavior_activations:
    peak = behavior_activations[beh].max()
    if peak > 0:
        behavior_activations[beh] /= peak

print("Behavior activation peaks (raw, before normalization):")
for beh, neuron_names in BEHAVIOR_GROUPS.items():
    traces = [motor_activations[n] for n in neuron_names if n in motor_activations]
    raw_peak = np.mean(traces, axis=0).max() if traces else 0
    any_spikes = any(len(motor_spike_trains[n]) > 0 for n in neuron_names)
    print(f"  {beh:12s}: peak={raw_peak:8.2f}  active={any_spikes}")

In [ ]:
# Plot behavior activations over time
t_ms = np.arange(num_steps) * dt

fig, axes = plt.subplots(len(BEHAVIOR_GROUPS), 1, figsize=(14, 10), sharex=True)
colors = plt.cm.Set2(np.linspace(0, 1, len(BEHAVIOR_GROUPS)))

for ax, (beh, trace), color in zip(axes, behavior_activations.items(), colors):
    ax.fill_between(t_ms, trace, alpha=0.6, color=color)
    ax.set_ylabel(beh, fontsize=9)
    ax.set_ylim(-0.05, 1.1)
    ax.axvline(SPONTANEOUS_MS, color="red", linestyle="--", alpha=0.5, label="stim onset")
    if ax == axes[0]:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle("Descending Neuron Behavioral Activations", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "behavior_activations.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Define Behavioral Action Primitives for Flybody

Each descending neuron group drives a stereotyped motor program. We define these as
CPG-like action patterns that modulate the 59-dimensional flybody action space.

**Action space layout** (walking config, wings disabled):
- `[0:6]` — Adhesion (6 claws)
- `[6:9]` — Head (abduct, twist, main)
- `[9:11]` — Abdomen (abduct, main)
- `[11:19]` — T1 left leg (coxa_abd, coxa_tw, coxa, fem_tw, fem, tib, tar, tar2)
- `[19:27]` — T1 right leg
- `[27:35]` — T2 left leg
- `[35:43]` — T2 right leg
- `[43:51]` — T3 left leg
- `[51:59]` — T3 right leg

In [ ]:
# Leg slice indices (each 8 DOFs: coxa_abd, coxa_tw, coxa, fem_tw, fem, tib, tar, tar2)
LEG_SLICES = {
    "T1_left":  slice(11, 19),
    "T1_right": slice(19, 27),
    "T2_left":  slice(27, 35),
    "T2_right": slice(35, 43),
    "T3_left":  slice(43, 51),
    "T3_right": slice(51, 59),
}
# Joint offsets within each leg slice
J_COXA_ABD, J_COXA_TW, J_COXA, J_FEM_TW, J_FEM, J_TIB, J_TAR, J_TAR2 = range(8)

# Tripod gait grouping
TRIPOD_1 = ["T1_left", "T2_right", "T3_left"]   # Move in phase
TRIPOD_2 = ["T1_right", "T2_left", "T3_right"]   # Antiphase

WALK_FREQ_HZ = 8.0  # Drosophila walking ~5-15 Hz

N_ACTIONS = 59  # Walking config


def tripod_walk_action(t_s, amplitude=1.0, freq=WALK_FREQ_HZ, direction=1.0):
    """Generate a tripod gait action vector at time t_s.

    direction: +1 = forward, -1 = backward
    amplitude: 0-1 modulation of gait intensity
    """
    action = np.zeros(N_ACTIONS)
    omega = 2.0 * np.pi * freq

    # Adhesion: maintain contact during stance phase
    for i in range(6):
        action[i] = 0.5  # Moderate adhesion always on

    for leg_group, phase_offset in [(TRIPOD_1, 0.0), (TRIPOD_2, np.pi)]:
        phase = omega * t_s + phase_offset
        # Stance fraction: sin > 0 → stance, sin < 0 → swing
        stance = np.sin(phase)

        for leg_name in leg_group:
            sl = LEG_SLICES[leg_name]
            a = amplitude

            # Coxa: drives forward/backward swing
            action[sl.start + J_COXA] = direction * a * 0.6 * np.sin(phase)
            # Femur: lift during swing (cos leads sin by π/2)
            action[sl.start + J_FEM] = a * 0.5 * (0.5 - 0.4 * np.cos(phase))
            # Tibia: extend during stance, flex during swing
            action[sl.start + J_TIB] = a * 0.3 * np.sin(phase + 0.3)
            # Tarsus: slight ground contact
            action[sl.start + J_TAR] = a * 0.2 * np.sin(phase)

    return action


def turning_action(t_s, amplitude=1.0, turn_sign=1.0, freq=WALK_FREQ_HZ):
    """Generate turning by biasing stride amplitude between sides.

    turn_sign: +1 = turn left (slow left, speed right), -1 = turn right
    """
    action = np.zeros(N_ACTIONS)
    omega = 2.0 * np.pi * freq

    for i in range(6):
        action[i] = 0.5

    for leg_group, phase_offset in [(TRIPOD_1, 0.0), (TRIPOD_2, np.pi)]:
        phase = omega * t_s + phase_offset
        for leg_name in leg_group:
            sl = LEG_SLICES[leg_name]
            # Reduce amplitude on the turning side
            is_left = "left" in leg_name
            side_scale = 0.3 if (is_left and turn_sign > 0) or (not is_left and turn_sign < 0) else 1.0
            a = amplitude * side_scale

            action[sl.start + J_COXA] = a * 0.6 * np.sin(phase)
            action[sl.start + J_FEM] = a * 0.5 * (0.5 - 0.4 * np.cos(phase))
            action[sl.start + J_TIB] = a * 0.3 * np.sin(phase + 0.3)

            # Coxa abduction bias for turning
            action[sl.start + J_COXA_ABD] = turn_sign * amplitude * 0.15

    return action


def escape_action(t_s, amplitude=1.0):
    """Giant fiber escape: rapid extension of all legs (jump-like)."""
    action = np.zeros(N_ACTIONS)
    for i in range(6):
        action[i] = 0.0  # Release adhesion for takeoff

    for leg_name, sl in LEG_SLICES.items():
        a = amplitude
        action[sl.start + J_FEM] = a * 1.5   # Strong femur extension
        action[sl.start + J_TIB] = a * 1.0   # Strong tibia extension
        action[sl.start + J_COXA] = a * 0.8  # Push backward
        action[sl.start + J_TAR] = a * 0.5   # Tarsus push

    return action


def grooming_action(t_s, amplitude=1.0, freq=3.0):
    """aDN1 grooming: front legs (T1) reach toward head in rhythmic sweeps."""
    action = np.zeros(N_ACTIONS)
    for i in range(6):
        action[i] = 0.5

    omega = 2.0 * np.pi * freq
    phase = omega * t_s
    a = amplitude

    for leg_name in ["T1_left", "T1_right"]:
        sl = LEG_SLICES[leg_name]
        # Reach forward and up toward head
        action[sl.start + J_COXA] = a * 1.2     # Swing forward
        action[sl.start + J_FEM] = a * 1.0      # Lift high
        action[sl.start + J_TIB] = a * 0.6 * np.sin(phase)  # Rhythmic sweep
        action[sl.start + J_TAR] = a * 0.3 * np.sin(phase)  # Tarsus sweep
        action[sl.start + J_COXA_ABD] = -a * 0.3  # Bring legs inward

    # Middle and hind legs hold stance
    for leg_name in ["T2_left", "T2_right", "T3_left", "T3_right"]:
        sl = LEG_SLICES[leg_name]
        action[sl.start + J_FEM] = 0.3
        action[sl.start + J_TIB] = 0.2

    return action


def feeding_action(t_s, amplitude=1.0, freq=2.0):
    """MN9 feeding: head/proboscis extension (mapped to head joints)."""
    action = np.zeros(N_ACTIONS)
    for i in range(6):
        action[i] = 0.5

    phase = 2.0 * np.pi * freq * t_s
    a = amplitude

    # Head extends downward (feeding posture)
    action[8] = -a * 0.3  # head: flex down

    # Slow body rocking
    action[10] = a * 0.2 * np.sin(phase)  # abdomen sway

    # Front legs reach forward slightly
    for leg_name in ["T1_left", "T1_right"]:
        sl = LEG_SLICES[leg_name]
        action[sl.start + J_COXA] = a * 0.4

    return action


print("Behavioral primitives defined:")
print("  forward  — tripod gait (P9/P9_oDN1)")
print("  turn     — asymmetric stride (DNa01/DNa02)")
print("  backward — reverse tripod (MDN)")
print("  escape   — full leg extension (Giant Fiber)")
print("  grooming — T1 head sweep (aDN1)")
print("  feeding  — head flexion (MN9)")

## 8. Map Neural Activity → Flybody Actions

For each flybody control step (2 ms), blend behavioral primitives weighted by the
corresponding descending neuron activation levels.

In [ ]:
CONTROL_DT_MS = 2.0  # Flybody control timestep
NEURAL_STEPS_PER_CONTROL = int(CONTROL_DT_MS / dt)  # = 20
N_CONTROL_STEPS = int(TOTAL_MS / CONTROL_DT_MS)

# Downsample behavior activations to control timestep
def downsample_activation(trace, neural_steps_per_control):
    """Average neural activation over each control timestep window."""
    n_control = len(trace) // neural_steps_per_control
    reshaped = trace[:n_control * neural_steps_per_control].reshape(n_control, neural_steps_per_control)
    return reshaped.mean(axis=1)

ctrl_activations = {}
for beh, trace in behavior_activations.items():
    ctrl_activations[beh] = downsample_activation(trace, NEURAL_STEPS_PER_CONTROL)

# Check if any motor neurons actually fired
any_motor_activity = any(
    len(steps) > 0 for steps in motor_spike_trains.values()
)

if not any_motor_activity:
    print("⚠ No motor neuron spikes detected in the simulation.")
    print("  This can happen with the PyTorch LIF model on the full 138K circuit")
    print("  (only ~0.1% of neurons fire). Generating synthetic activations")
    print("  to demonstrate the flybody integration.")
    print()

    # Synthetic fallback: ramp up forward walking after stim onset,
    # with brief escape burst
    stim_onset_ctrl = int(SPONTANEOUS_MS / CONTROL_DT_MS)
    for beh in ctrl_activations:
        ctrl_activations[beh] = np.zeros(N_CONTROL_STEPS)

    # Forward walking ramps up during stimulation
    ramp = np.linspace(0, 1, N_CONTROL_STEPS - stim_onset_ctrl)
    ctrl_activations["forward"][stim_onset_ctrl:] = ramp

    # Brief escape burst at stim onset
    burst_len = int(200 / CONTROL_DT_MS)  # 200 ms burst
    if "LC4s" in ACTIVE_EXPERIMENTS:
        burst = np.exp(-np.arange(burst_len) * CONTROL_DT_MS / 100.0)
        end = min(stim_onset_ctrl + burst_len, N_CONTROL_STEPS)
        ctrl_activations["escape"][stim_onset_ctrl:end] = burst[:end - stim_onset_ctrl]

    print("  Synthetic forward walking + escape response generated.")
else:
    print(f"Motor neuron activity detected — using simulation-driven activations.")

print(f"\nControl steps: {N_CONTROL_STEPS} ({CONTROL_DT_MS} ms each)")
for beh, trace in ctrl_activations.items():
    peak = trace.max()
    active_frac = (trace > 0.01).sum() / len(trace) * 100
    print(f"  {beh:12s}: peak={peak:.3f}  active={active_frac:.1f}%")

In [ ]:
# Precompute action sequences for TWO flies
# Fly 1: emphasizes forward walking + turning
# Fly 2: emphasizes escape + backward walking

action_seq_fly1 = np.zeros((N_CONTROL_STEPS, N_ACTIONS))
action_seq_fly2 = np.zeros((N_CONTROL_STEPS, N_ACTIONS))

for step in range(N_CONTROL_STEPS):
    t_s = step * CONTROL_DT_MS / 1000.0

    a_fwd = ctrl_activations["forward"][step]
    a_bwd = ctrl_activations["backward"][step]
    a_tl  = ctrl_activations["turn_left"][step]
    a_tr  = ctrl_activations["turn_right"][step]
    a_esc = ctrl_activations["escape"][step]
    a_grm = ctrl_activations["grooming"][step]
    a_fed = ctrl_activations["feeding"][step]

    # ── Fly 1: forward walking + turning + grooming ──
    act1 = np.zeros(N_ACTIONS)
    if a_fwd > 0.01:
        act1 += a_fwd * tripod_walk_action(t_s, amplitude=1.0, direction=1.0)
    if a_tl > 0.01:
        act1 += a_tl * turning_action(t_s, amplitude=1.0, turn_sign=1.0)
    if a_tr > 0.01:
        act1 += a_tr * turning_action(t_s, amplitude=1.0, turn_sign=-1.0)
    if a_grm > 0.01:
        act1 += a_grm * grooming_action(t_s, amplitude=1.0)
    if a_fed > 0.01:
        act1 += a_fed * feeding_action(t_s, amplitude=1.0)
    action_seq_fly1[step] = act1

    # ── Fly 2: escape + backward walking ──
    act2 = np.zeros(N_ACTIONS)
    if a_esc > 0.01:
        act2 += a_esc * escape_action(t_s, amplitude=1.0)
    if a_bwd > 0.01:
        act2 += a_bwd * tripod_walk_action(t_s, amplitude=1.0, direction=-1.0)
    # Also give fly2 some forward walking at reduced amplitude
    if a_fwd > 0.01:
        act2 += a_fwd * 0.5 * tripod_walk_action(t_s, amplitude=0.7, direction=1.0)
    action_seq_fly2[step] = act2

# Clip to valid action range
from flybody.fly_envs import template_task as _tt
_env_tmp = _tt(time_limit=0.1, disable_wings=True)
act_min = _env_tmp.action_spec().minimum
act_max = _env_tmp.action_spec().maximum
del _env_tmp

action_seq_fly1 = np.clip(action_seq_fly1, act_min, act_max)
action_seq_fly2 = np.clip(action_seq_fly2, act_min, act_max)

print(f"Fly 1 action sequence: {action_seq_fly1.shape}  (forward/turning/grooming)")
print(f"Fly 2 action sequence: {action_seq_fly2.shape}  (escape/backward/walking)")

## 9. Build Two-Fly Scene and Record Video

Two flies are placed in the same MuJoCo arena, each driven by different subsets of the
descending neuron activity from the same brain simulation:
- **Fly 1** (left): forward walking, turning, grooming (P9, DNa, aDN1 activations)
- **Fly 2** (right): escape response, backward walking (Giant Fiber, MDN activations)

In [ ]:
from dm_control import mjcf
from dm_control.locomotion.arenas import floors
from flybody.fruitfly.fruitfly import FruitFly
from flybody.tasks.constants import _WALK_PHYSICS_TIMESTEP, _WALK_CONTROL_TIMESTEP

# ── Build the action→ctrl permutation (adhesion-first action → MuJoCo ctrl order) ──
# Action order:  adhesion(6), head(3), abdomen(2), legs(48)
# MuJoCo order:  head(3), abdomen(2), legs(48), adhesion(6)
ACTION_TO_CTRL_LOCAL = np.concatenate([
    np.arange(53, 59),  # action[0:6]  adhesion → ctrl[53:59]
    np.arange(0, 3),    # action[6:9]  head     → ctrl[0:3]
    np.arange(3, 5),    # action[9:11] abdomen  → ctrl[3:5]
    np.arange(5, 53),   # action[11:59] legs    → ctrl[5:53]
])

# ── Create arena with two flies ──
arena = floors.Floor()
wk = dict(use_legs=True, use_wings=False, use_mouth=False, use_antennae=False,
          joint_filter=0.01, adhesion_filter=0.007,
          physics_timestep=_WALK_PHYSICS_TIMESTEP,
          control_timestep=_WALK_CONTROL_TIMESTEP)

fly1 = FruitFly(name='fly1', **wk)
fly2 = FruitFly(name='fly2', **wk)

# Attach fly1 at origin
pos1 = fly1.upright_pose.xpos.copy()
s1 = arena.mjcf_model.worldbody.add('site', pos=pos1)
fly1.create_root_joints(arena.attach(fly1, s1)); s1.remove()

# Attach fly2 offset to the right
FLY_OFFSET = np.array([0.6, 0.0, 0.0])
pos2 = fly2.upright_pose.xpos + FLY_OFFSET
s2 = arena.mjcf_model.worldbody.add('site', pos=pos2)
fly2.create_root_joints(arena.attach(fly2, s2)); s2.remove()

# Floor contact parameters
for geom in arena.ground_geoms:
    geom.friction = (0.5,)
    geom.solref = (0.001, 1)
    geom.solimp = (0.95, 0.99, 0.01)

# Visual settings
arena.mjcf_model.visual.map.znear = 0.001
arena.mjcf_model.visual.map.zfar = 50.0
arena.mjcf_model.statistic.extent = 4.01

# Close-up camera showing both flies
center = (pos1 + pos2) / 2
arena.mjcf_model.worldbody.add(
    'camera', name='duo_view',
    pos=[center[0] + 0.5, center[1] + 0.7, 0.35],
    xyaxes=[-0.8, 0.6, 0, -0.25, -0.33, 0.91],
    fovy=50,
)

# Compile physics
physics = mjcf.Physics.from_mjcf_model(arena.mjcf_model)

# Map each fly's actuators to global ctrl indices
fly1_ctrl = sorted(i for i in range(physics.model.nu) if 'fly1/' in physics.model.actuator(i).name)
fly2_ctrl = sorted(i for i in range(physics.model.nu) if 'fly2/' in physics.model.actuator(i).name)
fly1_global = np.array(fly1_ctrl)[ACTION_TO_CTRL_LOCAL]
fly2_global = np.array(fly2_ctrl)[ACTION_TO_CTRL_LOCAL]

# Find camera
duo_cam = next(i for i in range(physics.model.ncam) if 'duo' in physics.model.cam(i).name)

print(f"Two-fly scene compiled: {physics.model.nu} actuators ({len(fly1_ctrl)} + {len(fly2_ctrl)})")
print(f"Camera: duo_view (id={duo_cam})")

In [ ]:
# ── Run two-fly simulation and record video ──
RENDER_EVERY = 5
RENDER_HEIGHT = 480
RENDER_WIDTH = 640
PHYSICS_SUBSTEPS = 10  # physics_dt=2e-4, control_dt=2e-3 → 10 substeps

frames = []
positions_fly1 = []
positions_fly2 = []

print(f"Running two-fly simulation ({N_CONTROL_STEPS} control steps)...")
t0 = perf_counter()

for step in range(N_CONTROL_STEPS):
    # Apply actions for both flies
    physics.data.ctrl[fly1_global] = action_seq_fly1[step]
    physics.data.ctrl[fly2_global] = action_seq_fly2[step]

    # Step physics
    for _ in range(PHYSICS_SUBSTEPS):
        physics.step()

    # Track positions
    try:
        p1 = physics.named.data.xpos['fly1/thorax'].copy()
        p2 = physics.named.data.xpos['fly2/thorax'].copy()
    except Exception:
        p1 = p2 = np.zeros(3)
    positions_fly1.append(p1)
    positions_fly2.append(p2)

    # Render
    if step % RENDER_EVERY == 0:
        frames.append(physics.render(height=RENDER_HEIGHT, width=RENDER_WIDTH,
                                     camera_id=duo_cam))

    if (step + 1) % 500 == 0:
        print(f"  Step {step+1}/{N_CONTROL_STEPS}")

elapsed = perf_counter() - t0
print(f"\nDone in {elapsed:.2f}s — {len(frames)} frames rendered")

In [ ]:
# Save video as MP4
import imageio.v2 as imageio

video_path = OUTPUT_DIR / "flybody_two_flies.mp4"
effective_fps = 1000.0 / (CONTROL_DT_MS * RENDER_EVERY)
video_fps = min(effective_fps, 60.0)

imageio.mimwrite(str(video_path), frames, fps=video_fps)
print(f"Video saved: {video_path}")
print(f"  {len(frames)} frames at {video_fps:.0f} fps")
print(f"  Simulated time: {TOTAL_MS/1000:.1f}s  |  Playback: {len(frames)/video_fps:.1f}s")

In [ ]:
# Display video inline (Jupyter)
from flybody.utils import display_video
display_video(frames, framerate=int(video_fps))

## 10. Trajectory Analysis

In [ ]:
positions_fly1 = np.array(positions_fly1)
positions_fly2 = np.array(positions_fly2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top-down trajectory (x-y)
ax = axes[0]
t_ctrl = np.arange(len(positions_fly1)) * CONTROL_DT_MS
stim_idx = int(SPONTANEOUS_MS / CONTROL_DT_MS)

for pos, label, color in [(positions_fly1, 'Fly 1 (walk)', 'royalblue'),
                           (positions_fly2, 'Fly 2 (escape)', 'orangered')]:
    ax.plot(pos[:stim_idx, 0], pos[:stim_idx, 1],
            color=color, alpha=0.4, linewidth=1, linestyle='--')
    if stim_idx < len(pos):
        ax.plot(pos[stim_idx:, 0], pos[stim_idx:, 1],
                color=color, alpha=0.8, linewidth=1.5, label=f'{label} (stim)')
    ax.plot(pos[0, 0], pos[0, 1], 'o', color=color, markersize=6)
    ax.plot(pos[-1, 0], pos[-1, 1], '^', color=color, markersize=6)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Top-Down Trajectories')
ax.legend(fontsize=8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Height over time
ax2 = axes[1]
for pos, label, color in [(positions_fly1, 'Fly 1', 'royalblue'),
                           (positions_fly2, 'Fly 2', 'orangered')]:
    ax2.plot(t_ctrl[:len(pos)], pos[:, 2], color=color, linewidth=0.5, label=label)
ax2.axvline(SPONTANEOUS_MS, color='gray', linestyle='--', alpha=0.5, label='Stim onset')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Height Z (m)')
ax2.set_title('Body Height Over Time')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "trajectory_two_flies.png", dpi=150, bbox_inches="tight")
plt.show()

for name, pos in [("Fly 1", positions_fly1), ("Fly 2", positions_fly2)]:
    d = np.linalg.norm(pos[-1, :2] - pos[0, :2])
    print(f"{name} displacement: {d*1000:.2f} mm")

## 11. Save Spike Data for Reuse

In [ ]:
# Save spike data
spikes_path = OUTPUT_DIR / "spikes_integration.h5"
sort_idx = np.argsort(all_spike_times)

with h5py.File(spikes_path, "w") as f:
    pop = f.create_group("spikes/internal")
    ds = pop.create_dataset("timestamps", data=all_spike_times[sort_idx])
    ds.attrs["units"] = "ms"
    pop.create_dataset("node_ids", data=all_spike_ids[sort_idx].astype(np.uint64))
    pop.attrs["sorting"] = "by_time"

    # Save motor neuron spike trains separately
    motor_grp = f.create_group("motor_neurons")
    for name, steps in motor_spike_trains.items():
        times_ms = np.array(steps, dtype=np.float64) * dt
        motor_grp.create_dataset(name, data=times_ms)
        motor_grp[name].attrs["flywire_id"] = MOTOR_NEURONS[name]
        motor_grp[name].attrs["node_id"] = motor_node_ids[name]

    # Save experiment config
    meta = f.create_group("metadata")
    meta.attrs["spontaneous_ms"] = SPONTANEOUS_MS
    meta.attrs["stimulated_ms"] = STIMULATED_MS
    meta.attrs["total_ms"] = TOTAL_MS
    meta.attrs["experiments"] = str(ACTIVE_EXPERIMENTS)

# Save action sequence
np.save(OUTPUT_DIR / "action_sequence.npy", action_sequence)

print(f"Saved: {spikes_path}")
print(f"Saved: {OUTPUT_DIR / 'action_sequence.npy'}")
print(f"\nAll outputs in: {OUTPUT_DIR.resolve()}")